<a href="https://colab.research.google.com/github/garykbrixi/minerva/blob/main/examples/notebooks/loci_viewer_bokeh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Minerva loci viewer — interactive Bokeh

Same multimodal contacts as the static viewer (**base pairing** coral · **repeat** indigo · **protein** teal), but rendered with **Bokeh**: scroll to zoom to per-pixel detail, drag to pan, hover for position/value, and one **threshold slider per channel**. Per-position token-type tracks run along the top and left; the legend sits outside the plot.

Loads a **local checkpoint** from Google Drive (set `MODEL_DIR`). Fill the form, then `Runtime` → `Run all`.

In [ ]:
#@title Setup — mount Drive, load checkpoint, init Bokeh { display-mode: "form" }
import os, sys, gc, inspect, importlib.util
from pathlib import Path
import torch

IN_COLAB = importlib.util.find_spec("google.colab") is not None
MODEL_DIR = "/content/drive/MyDrive/sharing_minerva_gdrive"  #@param {type:"string"}
mount_drive = True  #@param {type:"boolean"}
install_flash_attn = False  #@param {type:"boolean"}
#@markdown Leave `install_flash_attn` off for the most robust Colab/T4 path; this uses torch SDPA.

if IN_COLAB:
    if mount_drive:
        from google.colab import drive
        drive.mount("/content/drive")
    !pip -q install "transformers>=4.41" "huggingface_hub>=0.23" safetensors biopython bokeh
    if install_flash_attn and importlib.util.find_spec("flash_attn") is None:
        !pip -q install flash-attn --no-build-isolation

from transformers import AutoTokenizer, AutoModelForMaskedLM

MODEL_DIR = os.path.abspath(os.path.expanduser(MODEL_DIR))
required = [
    "config.json",
    "model.safetensors",
    "tokenizer.json",
    "modeling_minerva.py",
    "jacobian.py",
    "constants.py",
    "visualization.py",
    "data.py",
]
missing = [name for name in required if not os.path.exists(os.path.join(MODEL_DIR, name))]
if missing:
    raise FileNotFoundError(
        f"MODEL_DIR does not look like the sharing_minerva_gdrive folder: {MODEL_DIR}\n"
        f"Missing: {missing}\n"
        "Set MODEL_DIR to the Google Drive path containing model.safetensors."
    )

# Avoid stale in-memory modules when rerunning cells after changing MODEL_DIR.
for _name in list(sys.modules):
    if (_name == "data" or _name == "visualization" or
        _name.startswith("transformers_modules.")):
        sys.modules.pop(_name, None)
for _old in ("model", "tokenizer", "sequence", "all_tokens"):
    globals().pop(_old, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

if MODEL_DIR not in sys.path:
    sys.path.insert(0, MODEL_DIR)

from data import extract_and_tokenize_gb
from visualization import head_contacts_rgb, bokeh_contact_viewer, save_bokeh_html
from bokeh.io import output_notebook, show
output_notebook()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = {"fp16": torch.float16, "bf16": torch.bfloat16, "fp32": torch.float32}['fp16']

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
model = AutoModelForMaskedLM.from_pretrained(
    MODEL_DIR,
    trust_remote_code=True,
    local_files_only=True,
    torch_dtype=DTYPE,
).to(DEVICE).eval()

_model_mod = inspect.getmodule(type(model))
_code_path = inspect.getfile(_model_mod)
if not install_flash_attn:
    _model_mod._HAS_FLASH = False
    _model_mod._HAS_FLASH_ROTARY = False

assert callable(bokeh_contact_viewer), "Loaded visualization code lacks bokeh_contact_viewer; update the sharing folder."
assert 'hidden_states.float()' in inspect.getsource(_model_mod.rmsnorm_func)
assert 'not _HAS_FLASH or x.device.type != "cuda"' in inspect.getsource(_model_mod.Attention.forward)

vocab = tokenizer.get_vocab()
nuc_ids = [vocab[c] for c in "atgc" if c in vocab]
aa_chars = list("ACDEFGHIKLMNPQRSTVWY")
aa_ids = [vocab[c] for c in aa_chars if c in vocab]

print("model_dir:", MODEL_DIR)
print("code:", _code_path)
print("device/dtype:", next(model.parameters()).device, next(model.parameters()).dtype)
print("flash-attn enabled:", _model_mod._HAS_FLASH, "rotary:", _model_mod._HAS_FLASH_ROTARY)
print("heads:", sorted(model.linear_heads.keys()))


In [ ]:
#@title 1. Choose locus and view { display-mode: "form", run: "auto" }
example  = "ug27"          #@param ["ug27", "twoayggay", "upload your own"]
view     = "heads (fast)"  #@param ["heads (fast)", "jacobian (detailed)", "both"]
head_set = "l2 (last-2)"   #@param ["l2 (last-2)", "l6 (last-6)"]
#@markdown Window in token positions. Leave both `0` for the preset/full locus.
record       = 0  #@param {type:"integer"}
window_start = 0  #@param {type:"integer"}
window_end   = 0  #@param {type:"integer"}
#@markdown Bokeh display
panel_px       = 760   #@param {type:"integer"}
prefilter      = 0.05  #@param {type:"number"}
init_threshold = 0.30  #@param {type:"number"}
jac_max_tokens = 384   #@param {type:"integer"}


In [ ]:
#@title Optional upload your own GenBank { display-mode: "form" }
UPLOADED = None
if IN_COLAB and example == "upload your own":
    from google.colab import files
    up = files.upload()
    if up:
        UPLOADED = list(up.keys())[0]
        print("uploaded:", UPLOADED)


In [ ]:
#@title 2. Load selected locus { display-mode: "form" }
def _local_example(name):
    p = os.path.join(MODEL_DIR, "examples", name)
    if not os.path.exists(p):
        raise FileNotFoundError(p)
    return p

PRESETS = {
    "ug27": dict(gb=_local_example("UG27_systems.gb"), record=2, window=(748, 1772)),
    "twoayggay": dict(gb=_local_example("TwoAYGGAY_Pseudomonas_fluorescens_SBW25.gb"), record=0, window=None),
}
if example == "upload your own":
    assert UPLOADED, "Run the upload cell first, or choose a built-in example."
    cfg = dict(gb=UPLOADED, record=record, window=None)
else:
    cfg = dict(PRESETS[example])
if window_end > window_start:
    cfg["window"] = (window_start, window_end)

records = extract_and_tokenize_gb(cfg["gb"], use_existing_translations=True)
record_obj = records[cfg["record"]]
sequence = record_obj["sequence"]
all_tokens = tokenizer.convert_ids_to_tokens(tokenizer.encode(sequence))
ntok = len(all_tokens)
window = cfg["window"]
suffix = "_l6" if head_set.startswith("l6") else ""
locus = record_obj["locus_name"][:44]
print("file:", cfg["gb"])
print("locus:", locus)
print("tokens:", ntok, "| window:", window or "(whole)", "| heads:", head_set, "")


In [ ]:
#@title 2. Heads view (interactive Bokeh) { display-mode: "form" }
if view.startswith("heads") or view == "both":
    import numpy as np
    heads = [f"base_pairing{suffix}", f"repeat{suffix}", f"protein{suffix}"]
    kw = dict(sequence=sequence, tokenizer=tokenizer, head_names=heads, return_dict=True)
    if window:
        kw.update(seed_start=window[0], seed_end=window[1])
    with torch.no_grad():
        pr = model.predict_contacts(**kw)["predictions"]
    chan = {h.replace(suffix, ""): pr[h].float().cpu().numpy() for h in heads}
    toks = all_tokens[window[0]:window[1]] if window else all_tokens
    layout = bokeh_contact_viewer(
        chan, tokens=toks,
        title=f"Heads {head_set} - {locus} {window or '(whole)'}",
        prefilter=prefilter, init_threshold=init_threshold, size=panel_px)
    show(layout)
    save_bokeh_html(layout, "heads_bokeh.html")
    print("saved heads_bokeh.html  (scroll = zoom, drag = pan, sliders = threshold)")


In [ ]:
#@title 3. Jacobian fingerprint view (interactive Bokeh, slow) { display-mode: "form" }
if view.startswith("jacobian") or view == "both":
    import time, numpy as np
    if window:
        js, je = window[0], min(window[1], window[0] + jac_max_tokens)
    else:
        c = ntok // 2
        js, je = max(0, c - jac_max_tokens // 2), min(ntok, c + jac_max_tokens // 2)
    print(f"jacobian window [{js}:{je}] = {je - js} tokens")
    t0 = time.time()
    fp = model.get_fingerprints(
        sequence, tokenizer, nuc_token_ids=nuc_ids, aa_token_ids=aa_ids,
        max_batch_size=32, position_range=(js, je), show_progress=True,
        autocast_dtype=DTYPE if DEVICE == "cuda" and DTYPE != torch.float32 else None,
        jac_aa_order=aa_chars)
    ch = {("base_pairing" if k == "basepairing" else k): np.asarray(v)
          for k, v in fp.channels.items() if k in ("basepairing", "repeat", "protein")}
    layout = bokeh_contact_viewer(
        ch, tokens=fp.tokens, title=f"Jacobian - {locus} [{js}:{je}]",
        prefilter=0.05, init_threshold=0.3, vmax=10.0, size=panel_px)
    print(f"done in {time.time() - t0:.0f}s")
    show(layout)
    save_bokeh_html(layout, "jacobian_bokeh.html")
    print("saved jacobian_bokeh.html")


---
**Notes**
- Interactive: mouse-wheel = zoom (per-pixel), drag = pan, hover = position + value, per-channel sliders raise/lower the contact threshold live.
- Top/left strips are the per-position token type (protein = teal, nucleotide = indigo, special = gray).
- Each view also writes a standalone `*.html` you can download and open offline (fully interactive, no Python needed).
- Heads view = one forward pass. Jacobian view is the slow, detailed one (window capped to `jac_max_tokens`, values scaled to `vmax=10`).
- For static publication PDFs, use `loci_viewer_hf.ipynb` instead.